# Model 2: Phishing URL Detection

**Project:** AI-Based Cybersecurity Threat Detection System  
**Dataset:** `PhiUSIIL_Phishing_URL_Dataset.csv`  
**Goal:** classify a URL as **Phishing (0)** or **Legitimate (1)** using URL-derived features.



## 1. Mount Google Drive and import libraries

We mount Drive so the notebook can read the dataset and save the trained model.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import joblib
import pandas as pd
import numpy as np
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

Mounted at /content/drive


## 2. Load and inspect the dataset

First inspect the shape, columns, labels, missing values, duplicates, and data types. The expected target column is `label`, and the URL column is `URL`.

In [2]:
folder = '/content/drive/MyDrive/ML_Project/'
dataset_path = folder + 'PhiUSIIL_Phishing_URL_Dataset.csv'
df = pd.read_csv(dataset_path)

print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Label counts:')
print(df['label'].value_counts(dropna=False))
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate URLs:', df['URL'].duplicated().sum())
print('Text columns:')
print(df.dtypes[df.dtypes == 'object'])

print('\nExample URLs labelled 1:')
print(df.loc[df['label'] == 1, 'URL'].sample(min(5, (df['label'] == 1).sum()), random_state=1).tolist())
print('\nExample URLs labelled 0:')
print(df.loc[df['label'] == 0, 'URL'].sample(min(5, (df['label'] == 0).sum()), random_state=1).tolist())

Dataset shape: (235795, 56)
Columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']
Label counts:
label
1    134850
0    100945


## 3. Check duplicate URLs and remove repeated copies

Before deduplication, check whether any URL is assigned more than one label. If conflicting labels exist, review them rather than assuming which label is correct. This notebook follows the original approach and keeps the first copy of each URL.

In [3]:
duplicate_urls = df[df['URL'].duplicated(keep=False)]
if len(duplicate_urls):
    print('Number of duplicated URL groups with each number of unique labels:')
    print(duplicate_urls.groupby('URL')['label'].nunique().value_counts().sort_index())
else:
    print('No duplicate URLs found.')

# Keep the first occurrence of each URL, as in the original notebook.
df = df.drop_duplicates(subset='URL').copy()
print('Shape after URL deduplication:', df.shape)

Number of duplicated URL groups with each number of unique labels:
label
1    425
Name: count, dtype: int64
Shape after URL deduplication: (235370, 56)


## 4. Select the features

We compare the full set of numeric features with the 13 URL-only features. We exclude text columns because the models in this notebook expect numeric inputs. URL-only features are used for the final model because they can be calculated from a URL entered by a user.

In [4]:
text_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
all_features = [col for col in df.columns if col not in text_cols + ['label']]
url_features = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfObfuscatedChar', 'NoOfLettersInURL',
    'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
    'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'IsHTTPS'
]

missing_features = [col for col in url_features if col not in df.columns]
if missing_features:
    raise ValueError(f'These expected URL features are missing from the dataset: {missing_features}')

# Confirm that the chosen model features are numeric.
non_numeric = df[url_features].select_dtypes(exclude=np.number).columns.tolist()
if non_numeric:
    raise ValueError(f'URL feature columns must be numeric: {non_numeric}')

y = df['label']
print('Number of all numeric features:', len(all_features))
print('Number of URL-only features:', len(url_features))
print('URL-only features:', url_features)

Number of all numeric features: 50
Number of URL-only features: 13
URL-only features: ['URLLength', 'DomainLength', 'IsDomainIP', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'IsHTTPS']


## 5. Compare all features with URL-only features

A stratified split keeps approximately the same label proportions in training and testing data. Accuracy measures overall correctness; F1 balances precision and recall for the positive class (label `1`, Legitimate).

In [5]:
comparison_results = []
for name, features in [('All numeric features', all_features), ('URL-only features', url_features)]:
    X_train, X_test, y_train, y_test = train_test_split(
        df[features], y, test_size=0.2, random_state=42, stratify=y
    )
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    comparison_results.append({
        'Feature set': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'F1 (label 1)': f1_score(y_test, predictions, pos_label=1)
    })

pd.DataFrame(comparison_results)

,Feature set,Accuracy,F1 (label 1)
0,All numeric features,1.000000,1.000000
1,URL-only features,0.997451,0.997779


## 6. Inspect the most important features

Decision Trees estimate feature importance based on how much each feature helps split the data. This is a model-specific measure, not proof that a feature causes phishing.

In [6]:
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    df[all_features], y, test_size=0.2, random_state=42, stratify=y
)
importance_model = DecisionTreeClassifier(random_state=42)
importance_model.fit(X_train_all, y_train_all)

feature_importance = pd.Series(
    importance_model.feature_importances_, index=all_features
).sort_values(ascending=False)
print(feature_importance.head(5))

URLSimilarityIndex    0.986448
LineOfCode            0.013139
IsHTTPS               0.000391
NoOfSubDomain         0.000022
DomainLength          0.000000
dtype: float64


## 7. Compare three models using URL-only features

We test Decision Tree, Random Forest, and Gaussian Naive Bayes on the same train/test split. The selected final model below remains a Decision Tree to match the original notebook.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df[url_features], y, test_size=0.2, random_state=42, stratify=y
)

models = [
    ('Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)),
    ('Naive Bayes', GaussianNB())
]

model_results = []
for name, model in models:
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    model_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'F1 (label 1)': f1_score(y_test, predictions, pos_label=1)
    })

results_df = pd.DataFrame(model_results).sort_values('F1 (label 1)', ascending=False)
results_df

,Model,Accuracy,F1 (label 1)
1,Random Forest,0.997578,0.997890
0,Decision Tree,0.997451,0.997779
2,Naive Bayes,0.913009,0.929273


## 8. Train and save the final model

This uses the Decision Tree model, as in your original code. The model is evaluated on the held-out test data before being saved. The saved feature list ensures the dashboard passes features in the correct order.

In [8]:
final_model = DecisionTreeClassifier(random_state=42)
final_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, test_predictions), 5))
print('F1 score for label 1 (Legitimate):', round(f1_score(y_test, test_predictions, pos_label=1), 5))
print('\nClassification report:')
print(classification_report(
    y_test, test_predictions,
    labels=[0, 1], target_names=['Phishing', 'Legitimate'], zero_division=0
))
print('Confusion matrix (rows=true, columns=predicted; labels 0, 1):')
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))

os.makedirs(folder, exist_ok=True)
joblib.dump(final_model, folder + 'model2_phishing.joblib')
joblib.dump(url_features, folder + 'model2_features.joblib')
print('\nSaved model:', folder + 'model2_phishing.joblib')
print('Saved features:', folder + 'model2_features.joblib')
print('Model file size:', round(os.path.getsize(folder + 'model2_phishing.joblib') / 1e6, 3), 'MB')

Accuracy: 0.99745
F1 score for label 1 (Legitimate): 0.99778

Classification report:
              precision    recall  f1-score   support

    Phishing       1.00      0.99      1.00     20104
  Legitimate       1.00      1.00      1.00     26970

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074

Confusion matrix (rows=true, columns=predicted; labels 0, 1):
[[19997   107]
 [   13 26957]]

Saved model: /content/drive/MyDrive/ML_Project/model2_phishing.joblib
Saved features: /content/drive/MyDrive/ML_Project/model2_features.joblib
Model file size: 0.072 MB


## 9. Extract the 13 features directly from a URL

This function lets the dashboard accept a URL string without requiring the user to supply a CSV row. Its feature calculations are intended to follow the definitions used in the original notebook. Feature matching is checked in the next section because even small differences can affect predictions.

In [9]:
def extract_features(url):
    url = str(url).strip()
    parsed = urlparse(url if '://' in url else 'http://' + url)
    domain = parsed.netloc.lower()

    # Work with the URL after removing the scheme and a leading www.
    rest = url.split('://', 1)[-1]
    if rest.startswith('www.'):
        rest = rest[4:]

    return {
        'URLLength': len(url) - (0 if url.endswith('/') else 1),
        'DomainLength': len(domain),
        'IsDomainIP': 1 if re.fullmatch(r'\d+\.\d+\.\d+\.\d+', domain) else 0,
        'NoOfSubDomain': domain.count('.') - 1,
        'HasObfuscation': 1 if '%' in url else 0,
        'NoOfObfuscatedChar': url.count('%'),
        'NoOfLettersInURL': len(re.findall(r'[A-Za-z]', rest)) - (0 if url.endswith('/') else 1),
        'NoOfDegitsInURL': len(re.findall(r'[0-9]', rest)),
        'NoOfEqualsInURL': url.count('='),
        'NoOfQMarkInURL': url.count('?'),
        'NoOfAmpersandInURL': url.count('&'),
        'NoOfOtherSpecialCharsInURL': len(re.findall(r'[^A-Za-z0-9=?&]', rest)),
        'IsHTTPS': 1 if url.lower().startswith('https://') else 0
    }

## 10. Check feature-extraction consistency

The dataset's URL features may be calculated with slightly different rules. This sample check compares our extracted values against the stored dataset features. Lower agreement means the extraction rules should be aligned more closely before relying on predictions for new URLs.

In [10]:
sample_size = min(5000, len(df))
sample = df.sample(sample_size, random_state=1)

agreement = {}
for col in url_features:
    extracted = sample['URL'].apply(lambda value: extract_features(value)[col])
    agreement[col] = round((extracted.to_numpy() == sample[col].to_numpy()).mean(), 3)

print(pd.Series(agreement, name='Exact agreement with dataset'))

URLLength                     0.842
DomainLength                  1.000
IsDomainIP                    1.000
NoOfSubDomain                 1.000
HasObfuscation                0.997
NoOfObfuscatedChar            0.995
NoOfLettersInURL              0.885
NoOfDegitsInURL               0.995
NoOfEqualsInURL               0.998
NoOfQMarkInURL                1.000
NoOfAmpersandInURL            0.981
NoOfOtherSpecialCharsInURL    0.930
IsHTTPS                       0.997
Name: Exact agreement with dataset, dtype: float64


## 11. Test URL-only predictions against known labels

This is a diagnostic check, not a replacement for the held-out evaluation above. It shows how predictions change when the model receives features extracted from raw URL text instead of the dataset's precomputed feature columns.

In [11]:
sample = df.sample(sample_size, random_state=1)
X_from_url = pd.DataFrame([extract_features(url) for url in sample['URL']])[url_features]
predictions_from_url = final_model.predict(X_from_url)
print('Agreement with labels on random sample using extracted features:',
      round(accuracy_score(sample['label'], predictions_from_url), 5))

test_rows = df.loc[X_test.index]
test_sample = test_rows.sample(min(5000, len(test_rows)), random_state=1)
X_test_from_url = pd.DataFrame([extract_features(url) for url in test_sample['URL']])[url_features]
test_predictions_from_url = final_model.predict(X_test_from_url)
print('Accuracy on held-out test rows using extracted URL features:',
      round(accuracy_score(test_sample['label'], test_predictions_from_url), 5))

Agreement with labels on random sample using extracted features: 0.9976
Accuracy on held-out test rows using extracted URL features: 0.9968


## 12. Try sample URLs

These are examples for demonstrating the prediction function. A prediction is only a model estimate, not a guarantee that a website is safe or malicious.

In [12]:
test_urls = [
    'https://www.google.com',
    'https://www.wikipedia.org',
    'http://secure-login-paypal.verify-account.xyz/login.php?id=123&user=abc',
    'http://192.168.4.7/bank/update/confirm.html'
]

for url in test_urls:
    features = pd.DataFrame([extract_features(url)])[url_features]
    prediction = int(final_model.predict(features)[0])
    label_name = 'Legitimate' if prediction == 1 else 'Phishing'
    print(url, '->', label_name)

https://www.google.com -> Legitimate
https://www.wikipedia.org -> Legitimate
http://secure-login-paypal.verify-account.xyz/login.php?id=123&user=abc -> Phishing
http://192.168.4.7/bank/update/confirm.html -> Phishing


## 13. Save the feature-extraction function for the dashboard

This creates `url_features.py` in the same project folder. The Streamlit app can import `extract_features` from this file.

In [13]:
feature_code = r'''import re
from urllib.parse import urlparse

def extract_features(url):
    url = str(url).strip()
    parsed = urlparse(url if '://' in url else 'http://' + url)
    domain = parsed.netloc.lower()
    rest = url.split('://', 1)[-1]
    if rest.startswith('www.'):
        rest = rest[4:]
    return {
        'URLLength': len(url) - (0 if url.endswith('/') else 1),
        'DomainLength': len(domain),
        'IsDomainIP': 1 if re.fullmatch(r'\d+\.\d+\.\d+\.\d+', domain) else 0,
        'NoOfSubDomain': domain.count('.') - 1,
        'HasObfuscation': 1 if '%' in url else 0,
        'NoOfObfuscatedChar': url.count('%'),
        'NoOfLettersInURL': len(re.findall(r'[A-Za-z]', rest)) - (0 if url.endswith('/') else 1),
        'NoOfDegitsInURL': len(re.findall(r'[0-9]', rest)),
        'NoOfEqualsInURL': url.count('='),
        'NoOfQMarkInURL': url.count('?'),
        'NoOfAmpersandInURL': url.count('&'),
        'NoOfOtherSpecialCharsInURL': len(re.findall(r'[^A-Za-z0-9=?&]', rest)),
        'IsHTTPS': 1 if url.lower().startswith('https://') else 0
    }
'''

with open(folder + 'url_features.py', 'w', encoding='utf-8') as file:
    file.write(feature_code)

print('Saved feature extraction function:', folder + 'url_features.py')

Saved feature extraction function: /content/drive/MyDrive/ML_Project/url_features.py


## Viva preparation: explain the notebook

- **What is phishing?** A deceptive website or URL designed to trick people into revealing information or taking unsafe actions.
- **What is the target label?** `0` means Phishing and `1` means Legitimate in this dataset/code.
- **Why use URL-only features?** They can be calculated from a URL typed into the dashboard, without needing the original dataset's extra columns.
- **Why split the data?** Training data teaches the model; test data checks performance on examples it did not train on.
- **What is stratification?** It keeps the class proportions similar in training and testing sets.
- **What is a Decision Tree?** A model that makes predictions by following a sequence of feature-based decisions.
- **What is Random Forest?** A collection of decision trees whose predictions are combined.
- **What is Naive Bayes?** A probabilistic classifier based on Bayes' theorem and simplifying assumptions about features.
- **What does F1 score show?** A combined measure of precision and recall; here the code reports F1 for label `1` (Legitimate).
- **Why save the model and feature list?** The Streamlit dashboard can load the trained model and pass the 13 inputs in the same order.
- **Important limitation:** high test accuracy does not by itself prove the system is reliable. Duplicate handling, feature extraction agreement, class-specific precision/recall, and possible data leakage should be reviewed before presenting performance claims.